# Build the model cache (run once, and again only when you change models)

Downloads everything the assistant needs into this notebook's output, so the daily notebook starts in about 1 minute instead of 5.

Settings: **Accelerator = None** (no GPU needed, saves quota), **Internet = On**. Then **Save Version → Save & Run All**.

Keep `LLM` the same as `LLM_MODEL` in `src/config.py`.

In [ ]:
import json, os, shutil, subprocess, sys, time, urllib.request
from datetime import date
from pathlib import Path

LLM = "qwen3.5:9b"  # keep in sync with src/config.py
PIP_PACKAGES = ["faster-whisper==1.2.1", "kokoro==0.9.4", "soundfile", "fastapi", "uvicorn",
                "cryptography", "kaggle", "sentence-transformers", "ddgs"]  # same as src/setup_env.py
OUT = Path("/kaggle/working/cache")
OUT.mkdir(parents=True, exist_ok=True)

try:
    urllib.request.urlopen("https://ollama.com", timeout=15)
    print("Internet: OK")
except OSError as e:
    raise SystemExit("NO INTERNET. In the right panel, set Internet = On "
                     "(needs a verified phone number), then Save & Run All again.") from e

In [ ]:
# 1. Ollama program + LLM weights
# The Ollama installer now needs "zstd" to unpack; Kaggle machines may not have it.
subprocess.run("command -v zstd || apt-get install -y -qq zstd "
               "|| (apt-get update -qq && apt-get install -y -qq zstd)", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
models = OUT / "ollama" / "models"
models.mkdir(parents=True, exist_ok=True)
env = {**os.environ, "OLLAMA_MODELS": str(models)}
server = subprocess.Popen(["ollama", "serve"], env=env)
for _ in range(60):  # wait until Ollama answers
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except OSError:
        time.sleep(1)
subprocess.run(["ollama", "pull", LLM], env=env, check=True)
server.terminate()

binary = Path(shutil.which("ollama")).resolve()
(OUT / "ollama" / "bin").mkdir(parents=True, exist_ok=True)
shutil.copy(binary, OUT / "ollama" / "bin" / "ollama")
shutil.copytree(binary.parent.parent / "lib" / "ollama", OUT / "ollama" / "lib" / "ollama",
                ignore=shutil.ignore_patterns("rocm*", "vulkan*"), dirs_exist_ok=True)
print("Ollama + model cached")

In [ ]:
# 2. Speech-to-text, text-to-speech and embedding models
from huggingface_hub import snapshot_download

hf = OUT / "hf"
snapshot_download("mobiuslabsgmbh/faster-whisper-large-v3-turbo", local_dir=hf / "whisper")
snapshot_download("hexgrad/Kokoro-82M", local_dir=hf / "kokoro",
                  allow_patterns=["config.json", "*.pth", "voices/*.pt"])
snapshot_download("intfloat/multilingual-e5-small", local_dir=hf / "embed",
                  ignore_patterns=["*.onnx", "onnx/*", "*.h5", "*.msgpack", "*.ot", "openvino/*"])
for d in hf.glob("*/.cache"):
    shutil.rmtree(d)

In [ ]:
# 3. ngrok program
(OUT / "ngrok").mkdir(exist_ok=True)
subprocess.run("curl -fsSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz "
               f"| tar -xz -C {OUT / 'ngrok'}", shell=True, check=True)

In [ ]:
# 4. Python packages that Kaggle does not already have
def freeze():
    out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    return {l for l in out.splitlines() if "==" in l}

before = freeze()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
new = sorted(freeze() - before)
print("caching:", new)
if new:
    subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--no-deps",
                    "-d", str(OUT / "wheels"), *new], check=True)

In [ ]:
# 5. Marker file the assistant looks for
(OUT / "PVA_MODELS_CACHE.json").write_text(json.dumps({"llm": LLM, "built": str(date.today())}))
size = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file()) / 1e9
print(f"Done: {size:.1f} GB in {OUT}")